In [1]:
using Pkg

# Use a fresh local first depot, but keep default depots for stdlib cache access
local_depot = joinpath(homedir(), "AppData", "Local", "JuliaDepotMasterarbeit_v2")
isdir(local_depot) || mkpath(local_depot)

default_depots = copy(DEPOT_PATH)
empty!(DEPOT_PATH)
push!(DEPOT_PATH, local_depot)
append!(DEPOT_PATH, filter(!=(local_depot), default_depots))
ENV["JULIA_DEPOT_PATH"] = join(DEPOT_PATH, ';')
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

Pkg.activate(".")
Pkg.instantiate()

required_packages = [
    "Statistics", "LinearAlgebra", "LsqFit", "LaTeXStrings", "Plots",
    "DataFrames", "Distributions", "JLD2", "DSP", "Interpolations",
    "Measurements", "NPZ", "Touchstone", "SavitzkyGolay",
    "Rotations", "SpecialPolynomials", "SpecialFunctions", "HDF5",
    "JSON", "Polynomials", "FFTW", "ProgressMeter"
 ]

installed = Set(keys(Pkg.project().dependencies))
missing = [pkg for pkg in required_packages if pkg ∉ installed]
if !isempty(missing)
    Pkg.add(missing)
    Pkg.instantiate()
else
    @info "All required packages are already in this project."
end

using Statistics
using LinearAlgebra
using LsqFit
using LaTeXStrings
using Plots
using Plots.Measures
using DataFrames
using Distributions
using Measurements
using NPZ
include("./../MADBead/MADBead.jl");

  Activating project at `c:\Users\nabil\OneDrive\Dokumente\Uni\Module\4. MS\Masterarbeit\spider-bead-pull`
┌ Info: All required packages are already in this project.
└ @ Main c:\Users\nabil\OneDrive\Dokumente\Uni\Module\4. MS\Masterarbeit\spider-bead-pull\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W0sZmlsZQ==.jl:32


In [2]:
data = npzread("./data/ET_results.npz")

ET_red      = permutedims(data["ETs_reduced"], (2, 1))       # (n_z_reduced, n_freq), complex
ET_full     = permutedims(data["ETs_full"], (2, 1))          # (n_z_full, n_freq), complex
z_red       = vec(data["z_reduced"])    # (n_z_reduced,), meters
z_full      = vec(data["z_full"])       # (n_z_full,), meters
ν           = vec(data["frequencies"])  # (n_freq,), Hz

println("Reduced: $(length(z_red)) points, $(length(ν)) frequencies")
println("Full:    $(length(z_full)) points, $(length(ν)) frequencies")

Reduced: 4 points, 1001 frequencies
Full:    17 points, 1001 frequencies


In [3]:
rel_err = 5e-2

ET_red_err = rel_err .* ET_red
ET_full_err = rel_err .* ET_full

ET_red_meas = real.(ET_red) .± real.(ET_red_err) .+ 1im .* (imag.(ET_red) .± imag.(ET_red_err))
ET_full_meas = real.(ET_full) .± real.(ET_full_err) .+ 1im .* (imag.(ET_full) .± imag.(ET_full_err))

1001×17 Matrix{Complex{Measurement{Float64}}}:
     (7.95±0.4)+(0.914±0.046)im  …     (4.34±0.22)-(2.58±0.13)im
  (-11.78±0.59)+(2.38±0.12)im           (5.1±0.25)-(1.288±0.064)im
     (5.99±0.3)-(5.96±0.3)im           (3.45±0.17)-(3.49±0.17)im
     (4.06±0.2)-(10.42±0.52)im       (0.225±0.011)+(5.1±0.26)im
   (0.46±0.023)+(5.64±0.28)im       (-0.943±0.047)-(5.24±0.26)im
   (-5.33±0.27)-(9.47±0.47)im    …    (-3.17±0.16)-(4.84±0.24)im
    (-8.9±0.44)-(6.29±0.31)im          (2.56±0.13)+(2.7±0.13)im
   (-7.26±0.36)-(3.48±0.17)im          (5.61±0.28)+(0.258±0.013)im
   (-8.12±0.41)+(2.9±0.14)im          (-9.38±0.47)+(1.269±0.063)im
   (-4.52±0.23)+(5.47±0.27)im          (6.21±0.31)-(7.5±0.37)im
             ⋮               ⋱  
  (-1.61±0.081)-(2.96±0.15)im        (0.439±0.022)+(5.84±0.29)im
     (2.3±0.12)+(0.67±0.034)im         (4.13±0.21)+(5.87±0.29)im
    (2.49±0.12)+(1.541±0.077)im        (5.74±0.29)+(2.81±0.14)im
   (-3.21±0.16)+(0.591±0.03)im   …     (-7.4±0.37)-(0.614±0.031)im
    (

In [6]:
ϵ_b = 9.23 ± 0.2
r_b = (2.93e-3 ± 15e-6)/2;#Grade 28
σ_Al = (5.0 ± 1)*1e7
tanD_d = 1e-5 ± 1e-6
ϵ_d = 9.3 ± 0.1
d_d = (1 ± 0.05)*1e-3

δ_e_mie = [MADBead.calc_δ_e_mie(ϵ_b,r_b;f=f) for f in ν];
δ_c_mie = [MADBead.calc_δ_c_mie(ϵ_b,r_b;f=f) for f in ν];

z_m_0 = 675.551e-3 ± 1e-3
σ_m_0 = σ_Al
d_v_i = [8.265 ± 0.1, 9.813 ± 0.1, 9.813 ± 0.1]*1e-3
E0_0 = 1 ± 1
n_disk = 3 ± 0

N_mc = 100
idx_sub_test = 1:1:length(ν)

p0_all = Dict("E_0"=>E0_0,"z_m"=>z_m_0,"σ_m"=>σ_m_0,"n_disk"=>n_disk)
for i in 1:Int(n_disk.val)
    p0_all["d_v_$i"]=d_v_i[i]
    p0_all["d_d_$i"]=d_d
    p0_all["ϵ_d_$i"]=ϵ_d
    p0_all["tanD_d_$i"]=tanD_d
end
p0_all["r_b"]=r_b
p0_all["ϵ_b"]=ϵ_b

keys_optim = ["E_0","d_v_1","d_v_2","d_v_3"];
keys_helper = ["n_disk"]
keys_fixed = [setdiff(keys(p0_all),keys_optim,keys_helper)...];
keys_all = [keys_optim...,keys_helper...,keys_fixed...];

In [7]:
p_all_ν_mc_red = MADBead.fit_E_z_MC(ν[idx_sub_test],z_red,ET_red_meas[idx_sub_test,:],p0_all,keys_optim,keys_fixed,keys_helper,N_mc);
p_final_ν_red = DataFrame("f"=>ν);
[p_final_ν_red[!,key].= (mean.(p_all_ν_mc_red[!,key]).±std.(p_all_ν_mc_red[!,key])) for key in keys(p0_all)];

int_dz_E_mc_red = zeros(Complex{Float64},length(ν),N_mc);
for f in 1:length(ν), i in 1:N_mc
    p_dict_red = Dict(key=>p_all_ν_mc_red[f,key][i] for key in keys(p0_all))
    int_dz_E_mc_red[f,i] = MADBead.int_dz_E_param(p_dict_red;f=ν[f])
end

In [ ]:
p_all_ν_mc_full = MADBead.fit_E_z_MC(ν[idx_sub_test],z_full,ET_full_meas[idx_sub_test,:],p0_all,keys_optim,keys_fixed,keys_helper,N_mc);
p_final_ν_full = DataFrame("f"=>ν);
[p_final_ν_full[!,key].= (mean.(p_all_ν_mc_full[!,key]).±std.(p_all_ν_mc_full[!,key])) for key in keys(p0_all)];

int_dz_E_mc_full = zeros(Complex{Float64},length(ν),N_mc);
for f in 1:length(ν), i in 1:N_mc
    p_dict_full = Dict(key=>p_all_ν_mc_full[f,key][i] for key in keys(p0_all))
    int_dz_E_mc_full[f,i] = MADBead.int_dz_E_param(p_dict_full;f=ν[f])
end

In [ ]:
int_dz_E_mc_full = zeros(Complex{Float64},length(ν),N_mc);
for f in 1:length(ν), i in 1:N_mc
    p_dict_full = Dict(key=>p_all_ν_mc_full[f,key][i] for key in keys(p0_all))
    int_dz_E_mc_full[f,i] = MADBead.int_dz_E_param(p_dict_full;f=ν[f])
end

In [ ]:
freq_idx = argmax(abs.(getfield.(p_final_ν[!,"E_0"],:val))) #+200
mc_idx = 1#109+1
p_select = Dict(key=>p_all_ν_mc[freq_idx,key][mc_idx] for key in keys(p0_all))

distance_fit,ignore,ignore = MADBead.fit_param_to_model(p_select);
z_σ = sum(distance_fit) .+ [0,1e-3]
z_ϵ = [sum(distance_fit[2*i+1:end]) .+ [0,-1e-3] for i in 1:Int(n_disk.val)]

z_pos_fit = z_pos[1]-0.04:1e-4:z_pos[end]+0.01

plot()
vspan!(z_σ*1e3,label="",c="darkgrey")
vspan!(z_ϵ*1e3,c="lightgrey",label="")

scatter!(z_pos*1e3,abs.(∫dA_E[idx_sub_test[freq_idx],:]),msw=1,c=2,label="Measurement")

δ_mie_select = real.(δ_e_mie[freq_idx]).val + 1im*imag.(δ_e_mie[freq_idx]).val
δc_mie_select = real.(δ_c_mie[freq_idx]).val + 1im*imag.(δ_c_mie[freq_idx]).val

fit_plot = sqrt.(abs.(MADBead.E_field2_conv_1D_z_param(z_pos_fit,p_select,f=ν[freq_idx],δ=δ_mie_select,δc=δc_mie_select)))

plot!(z_pos_fit*1e3,fit_plot,c=4,ls=:dash,label="Fit")

deconv_plot = abs.(MADBead.E_field_1D_z_param(z_pos_fit,p_select,f=ν[freq_idx]))

plot!(z_pos_fit*1e3,deconv_plot,c=3,ls=:dashdot,label="Deconvoluted")

plot!(xlabel=L"z \:\mathrm{[mm]}",ylabel=L"E_{1D} \:\: \mathrm{[Vm]}")
plot!()